In [1]:
#FastText - word2tec

import pandas as pd
import re
from gensim.models import FastText
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from nltk.corpus import stopwords
import nltk
import numpy as np


# Stopwords setini indir
nltk.download('stopwords')
STOPWORDS = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\yusuf\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
# Remove special characters, punctuation, and convert to lowercase
def preprocess_text(text):
    if not isinstance(text, str):
        return ''  # Return empty string for NaN or non-string values
    text = re.sub(r'[^\w\s]', '', text.lower())  # Remove punctuation and convert to lowercase
    text = ' '.join(word for word in text.split() if word not in STOPWORDS)  # Remove stopwords
    return text

In [3]:

# Load the dataset
file_path = "filtered_file_2.xlsx"  # Replace with your actual file path
df = pd.read_excel(file_path)

# Apply text preprocessing
df['Abstract_clean'] = df['Abstract'].apply(preprocess_text)
df['Research_Areas_clean'] = df['Research Areas'].apply(preprocess_text)

# Combine text columns for analysis
df['combined_text'] = df['Abstract_clean'] + ' ' + df['Research_Areas_clean']

# Tokenize the text for FastText training
df['tokenized_text'] = df['combined_text'].apply(lambda x: x.split())

# Train FastText model
fasttext_model = FastText(
    sentences=df['tokenized_text'],
    vector_size=300,  # Size of the word vectors
    window=5,         # Context window size
    min_count=5,      # Minimum frequency for a word to be included
    sg=1,             # Skip-gram model
    epochs=10         # Number of training epochs
)

In [5]:
# Function to get sentence vectors
def get_sentence_vector(sentence, model):
    if not isinstance(sentence, str):
        return np.zeros(model.vector_size)
    words = sentence.split()
    word_vectors = [model.wv[word] for word in words if word in model.wv]
    if len(word_vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(word_vectors, axis=0)

In [10]:
# User query
user_input = "cell cancer bio bioinformatics machine learning algorithm human life gene gen heredity"
user_input_clean = preprocess_text(user_input)
user_vector = get_sentence_vector(user_input_clean, fasttext_model)

# Compute similarity scores
df['vector'] = df['combined_text'].apply(lambda x: get_sentence_vector(x, fasttext_model))
df['similarity'] = df['vector'].apply(lambda x: cosine_similarity([user_vector], [x])[0][0])

# Calculate article count per researcher
df['article_count'] = df['Researcher Ids'].str.split(';').apply(len)

# Flatten researchers for grouping
def split_researchers(row):
    if not isinstance(row['Researcher Ids'], str):
        return pd.DataFrame()
    researchers = row['Researcher Ids'].split(';')
    data = []
    for researcher in researchers:
        data.append({
            'Researcher': researcher.strip(),
            'combined_text': row['combined_text'],
            'Publication Year': row['Publication Year'],
            'similarity': row['similarity'],
            'article_count': row['article_count']
        })
    return pd.DataFrame(data)

flattened_df = pd.concat(df.apply(split_researchers, axis=1).to_list(), ignore_index=True)

# Group by researcher and calculate average similarity
grouped_df = flattened_df.groupby('Researcher', as_index=False).agg({
    'similarity': 'mean',
    'article_count': 'sum',
    'Publication Year': 'max'
})

# Count the number of articles per researcher
article_counts = flattened_df['Researcher'].value_counts()
grouped_df['article_count'] = grouped_df['Researcher'].map(article_counts)



In [9]:
# Normalize similarity scores based on the number of articles
k = 0.184  # Growth rate, calculated to make f(25) approximately 1
grouped_df['normalized_similarity'] = grouped_df['similarity'] * (1 - np.exp(-k * grouped_df['article_count']))

# Scale year weights for stronger impact (higher weight for recent years)
# Year set to 2026 because the articles published 2025 are being 2026 in the dataset
current_year = 2026
year_diff = current_year - grouped_df['Publication Year']
a=1
grouped_df['year_weight'] =  0.75 + 0.2 * (1 - 1 / (1 + np.exp(-a * (year_diff - 5))))

# Combine normalized similarity and year weight into a final score
grouped_df['final_score'] = grouped_df['normalized_similarity'] * grouped_df['year_weight']
# Sort researchers by final score
top_matches = grouped_df.sort_values(by='final_score', ascending=False).head(30)

# Display the top researchers
print(top_matches[['Researcher', 'final_score', 'article_count', 'year_weight', 'Publication Year']])

                               Researcher  final_score  article_count  \
2626            Birant, Derya/U-6211-2017     0.726732             66   
11488       Nasibov, Efendi/ISV-0173-2023     0.701877             46   
4243         DALKILIC, GOKHAN/B-8292-2014     0.699002             29   
15913           UTKU, Semih/LGY-2879-2024     0.698795             21   
14182           Selver, Alper/V-5118-2019     0.698777             42   
12293       Ozsoydan, Fehmi/AAC-5530-2019     0.696678             32   
15519      Topaloglu, Şeyda/AAB-3269-2021     0.690775             39   
2251         Baykasoglu, Adil/G-4311-2010     0.690693            129   
14831          Subulan, Kemal/P-2090-2019     0.690630             29   
2627    Birant, Kökten Ulaş/AAD-8386-2022     0.686702             16   
6645            Golcuk, Ilker/B-2116-2015     0.683865             19   
9758             Kut, Recep/AAC-3485-2019     0.681015             21   
15382            Tezel, Baris/P-2770-2019     0.679